# Controller evaluation and IS2 extensions

Generated notebook for the GridMind IS2 project. Run from the project root or use the first cell to configure the path.

In [ ]:
from pathlib import Path
import sys
from importlib import import_module
ROOT=Path.cwd()
if not (ROOT/'config.py').exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT))
source=import_module('01_data_acquisition.data_source'); prep=import_module('02_data_preprocessing.preprocess'); ga=import_module('06_genetic_optimizer.ga_optimizer'); fuzzy=import_module('08_evaluation.fuzzy_logic'); evaluation=import_module('08_evaluation.metrics')

In [ ]:
raw=source.generate_building_data(days=5,seed=42)
clean=prep.clean_and_engineer(raw)
window=clean.tail(24); load=window.load_kw.to_numpy(); solar=window.solar_kw.to_numpy(); prices=window.price_per_kwh.to_numpy()

 Compare matched-day energy-cost proxy with optimized schedule objective; report the distinction.
no_battery=float((load-solar).clip(min=0).dot(prices))
optimized=ga.optimize_battery_schedule(load,solar,prices,seed=42,population_size=40,generations=25)
{'no_battery_energy_cost_proxy':no_battery,'ga_objective_including_penalties':optimized['cost']}

 Fuzzy logic output is interpretable and bounded between 0 and 1.
priority=fuzzy.fuzzy_cooling_priority(float(window.outdoor_temp_c.iloc[-1]),float(window.occupancy.iloc[-1]),float(window.price_per_kwh.iloc[-1]))
{'cooling_priority':priority}

 For a rigorous comparison, run each controller on the exact same profile and initial SOC.
# Required methods: No optimization, Rule-based, GA-only, PPO-only, GA+PPO.
# Repeat across multiple seeds and report mean ± standard deviation; do not use illustrative PDF values.